# Setup and Data Loading:

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scrapbook as sb
import pandas as pd
from utils import *

def get_combined_val_metric_df(agg_df:pd.DataFrame) -> pd.DataFrame:
    def get_val_metric_df(agg_df:pd.DataFrame, idx:int=0, postfix:str='') -> pd.DataFrame:
        val_metrics_idx = pd.DataFrame(index=agg_df.index, data=list(agg_df['val_metrics'].apply(lambda x: x[idx])))
        val_metrics_idx.columns = val_metrics_idx.columns.str.replace(f'/dataloader_idx_{idx}', postfix).str.replace('val_', '')
        return val_metrics_idx

    agg_df = agg_df[agg_df['val_metrics'].notna()] # TODO: delete this line
    val_metrics_short = get_val_metric_df(agg_df, idx=0)
    val_metrics_long = get_val_metric_df(agg_df, idx=1, postfix='_long')
    val_metrics = val_metrics_short.join(val_metrics_long, how='outer').sort_index(axis=1)
    return val_metrics

# create post-processed aggregate dataframe of all the notebooks' metrics
def load_notebook_df(dir:str, simple:bool=True) -> pd.DataFrame:
    book = sb.read_notebooks(dir) # create a scrapbook named `book`
    nb_dfs = [nb.scrap_dataframe for nb in book.notebooks]
    if len(nb_dfs)==0: return pd.DataFrame()
    agg_df: pd.DataFrame = pd.concat(nb_dfs)
    agg_df['experiment_name'] = agg_df['filename'].str.replace('.ipynb', '')
    agg_df = agg_df.pivot(index='experiment_name', columns='name', values='data')
    if simple: agg_df = agg_df.drop(columns=['checkpoint_path', 'uq_over_time'], errors='ignore')
    agg_df = agg_df.apply(lambda col: col.astype(col.dropna().dtype), axis=1) # correct dtypes

    agg_df['Bayesian'] = ~agg_df['LPPC_val'].isna() # record if Bayesian model?
    val_metrics = get_combined_val_metric_df(agg_df)
    agg_df = agg_df.drop(columns='val_metrics')
    agg_df = agg_df.join(val_metrics)

    crashed_notebooks = list(agg_df[agg_df.select_dtypes(include='float').isna().all(axis=1)].index)
    if len(crashed_notebooks) > 0:
        print('The following notebooks failed to evaluate at all (due to errors):')
        display(crashed_notebooks)
        agg_df = agg_df.drop(crashed_notebooks)

    return agg_df

import os # Verified to work on 3/9/26
def load_notebooks_df_and_handle_failures(dir:str) -> pd.DataFrame:
    ''' loads all notebooks and automatically archives failures '''
    agg_df = load_notebook_df(dir)

    # convert all object columns (e.g. NaN and Inf) to float (technically handling failure)
    cols = agg_df.select_dtypes(include=['object']).columns
    agg_df[cols] = agg_df[cols].apply(pd.to_numeric, errors='coerce')

    # pretty safe to assume failure beyond this threshold
    agg_df['Failure'] = agg_df['mse_log_energy_spectrum_0.2_FTT_samplewise'].astype(float) > 0.4
    #agg_df['Failure'] |= agg_df['xcor_mse'] > 1e-3 # seems not necessary?

    # archive NEW failures (naively archiving new+old can cause edge case bugs for duplicate notebook names)
    os.system(f'mkdir {dir}/failures 2> /dev/null')
    failure_notebook_fns = f'{dir}/' + agg_df[agg_df['Failure']].index + '.ipynb'
    for fn in failure_notebook_fns: os.system(f'mv {fn} {dir}/failures/. 2> /dev/null')

    try: # load previous failures
        agg_df_failures = load_notebook_df(f'{dir}/failures')
        if agg_df_failures.empty: raise FileNotFoundError # prevent edge case bug
        agg_df_failures['Failure'] = True
        agg_df = pd.concat([agg_df, agg_df_failures])
    except FileNotFoundError: pass

    return agg_df

In [ ]:
agg_df = load_notebooks_df_and_handle_failures('notebook_runs/')
agg_df_VI = agg_df[agg_df['Bayesian']]
agg_df_MLE = agg_df[~agg_df['Bayesian']]
agg_df_MLE = agg_df_MLE.iloc[:, ~agg_df_MLE.isna().all(axis=0).values] # drop Bayesian columns
print(f'{len(agg_df)=}')
agg_df.head()

In [ ]:
agg_df.columns # show all columns

In [ ]:
agg_df['Failure'].value_counts()

# Normalize and Create Summary Metrics:

In [ ]:
from typing import Callable
def mutate_matching_columns(df:pd.DataFrame, like_patterns:list[str]|str, func:Callable, prefix:str='', **kwargs) -> pd.DataFrame:
    if isinstance(like_patterns, str): like_patterns = [like_patterns]
    matching_columns = df.columns[np.logical_or.reduce([df.columns.str.contains(like_pattern) for like_pattern in like_patterns])]
    print(f'Applying {func.__name__} to {len(matching_columns)} columns:\n {matching_columns}')
    df[matching_columns] = df[matching_columns].apply(func, **kwargs)
    df.rename(columns={col: prefix+col for col in matching_columns}, inplace=True)
    return df

# normalize all the columns
normalized_loss_df = agg_df.select_dtypes(include='float').dropna(how='all')
normalized_loss_df = mutate_matching_columns(normalized_loss_df, 'xcor.*delta', lambda x: x.abs(), prefix='abs_')
normalized_loss_df = normalized_loss_df.apply(np_normalize)

# make everything into a "loss"
normalized_loss_df = mutate_matching_columns(normalized_loss_df, ['LPPC', 'R\^2'], lambda x: -x, prefix='neg_')

print('normalized_loss_df.columns:\n', normalized_loss_df.columns)
display(normalized_loss_df)

## Summary of Findings:
* TS=16 might be the best in general? it's best of Everything_TS=N and Everything_TS=N_E=0.25 comparisons also Everything_TS=16 has the 3rd best energy spectrum, and Everything_TS=16_E=0.25 had really promising Xcor (2/3/26)
* average metric and worst metric summaries are still bad because they aggregate unimportant metrics 2/9/26
* 

In [ ]:
# for average and std, only use important columns
#important_loss_names = ['ECE', 'neg_LPPC_val', 'neg_LPPC_val_long', 'mse_bulk_velocity', 'mse_log_energy_spectrum',
#                        'mse_rms', 'xcor_mse', 'xcor_mse_cum', 'xcor_mse_last', 'neg_R^2', 'neg_R^2_long',
#                        'last_TS_neg_R^2', 'last_TS_neg_R^2_long', 'sMAPE', 'sMAPE_long', 'MAE', 'MAE_long']
important_loss_names = []
def add_important_loss_names(important_loss_prefixes, postfixes=['']):
    for loss_prefix in important_loss_prefixes:
        for postfix in postfixes:
            important_loss_names.append(f'{loss_prefix}{postfix}')

# add various flow stats
flow_stats_loss_names = ['mse_log_energy_spectrum', 'mse_bulk_velocity', 'mse_rms']
postfixes = ['_0.2_FTT_samplewise', '_1_FTT_MAP', '_1_FTT_samplewise']
add_important_loss_names(flow_stats_loss_names, postfixes=postfixes)

# add various xcor metrics
xcor_loss_names = ['xcor_mse', 'abs_xcor_peak_magnitude_delta', 'abs_xcor_peak_loc_delta']
add_important_loss_names(xcor_loss_names, postfixes=postfixes)

# down-select to only the important losses
important_normalized_loss_df = normalized_loss_df[list(set(important_loss_names))].dropna(how='all')

summary_metrics = pd.DataFrame(index=important_normalized_loss_df.index)
summary_metrics['worst_loss_name'] = important_normalized_loss_df.apply(lambda row: row.idxmax(skipna=True), axis=1)
summary_metrics['worst_loss'] = important_normalized_loss_df.apply(lambda row: row.max(skipna=True), axis=1)
summary_metrics['average_loss'] = important_normalized_loss_df.apply(lambda row: row.mean(skipna=True), axis=1)
summary_metrics['loss_std'] = important_normalized_loss_df.apply(lambda row: row.std(skipna=True), axis=1)
summary_metrics = summary_metrics.sort_values(by='worst_loss')
print('='*50+'\nGOTCHA: average and std are only computed on important_loss_names!', flush=True)
print('important_loss_names:')
display(important_loss_names)
print("="*50)
display(summary_metrics)

In [ ]:
import matplotlib.pyplot as plt
normalized_losses = important_normalized_loss_df.values.flatten()
normalized_losses = normalized_losses[~np.isnan(normalized_losses)]
plt.hist(normalized_losses, bins=100)
plt.locator_params(axis='x', nbins=10)
plt.title('Distribution of important_normalized_losses')
plt.show()

# Is MLE or VI better?
* It appears that (at least for this sample) **VI is better at xcor?!** \
(This is not a scientific conclusion! just a guess...)
* Actually a more direct head-to-head comparison *shows that MLE is likely better* (when comparing sample xcor).
  * mse_self_xcor MLE=2.583145908932151e-05; VI=3.18412450736564e-05
  * **However still not certain because that would require MAP prediction display!**

In [ ]:
print('computed on sample predictions then averaged:')
print(f'VI_xcor_mse = {agg_df_VI["mse_log_energy_spectrum_1_FTT_samplewise"].mean()}')
print(f'MLE_xcor_mse = {agg_df_MLE["mse_log_energy_spectrum_1_FTT_samplewise"].mean()}')

print('computed on MAP predictions:')
print(f'VI_xcor_mse = {agg_df_VI["mse_log_energy_spectrum_1_FTT_MAP"].mean()}')
print(f'MLE_xcor_mse = {agg_df_MLE["mse_log_energy_spectrum_1_FTT_MAP"].mean()}')


In [ ]:
print('computed on MAP predictions then averaged:')
print('VI-compare: Everything_TS=4 ', agg_df.loc['Everything_TS=4']['mse_log_energy_spectrum_1_FTT_MAP'])
print('MLE-compare: MLE_TS=4 ', agg_df.loc['MLE_TS=4']['mse_log_energy_spectrum_1_FTT_MAP'])

In [ ]:
print('computed on MAP predictions then averaged:')
print('VI-compare: Everything_TS=4_no_uq_prop ', agg_df.loc['Everything_TS=4_no_uq_prop']['mse_log_energy_spectrum_1_FTT_MAP'])
print('MLE-compare: MLE_TS=4_dropout=0.2 ', agg_df.loc['MLE_TS=4_dropout=0.2']['mse_log_energy_spectrum_1_FTT_MAP'])

# Looking at Best Experiments:

In [ ]:
agg_df_VI.sort_values(by='LPPC_val', ascending=False).head()

Unfortunately PP-LL isn't predictive of good flow stats (in particular xcor is bad for Everything_TS=4_gaps_reprod)...

In [ ]:
agg_df_VI.sort_values(by='LPPC_val_long', ascending=False).head()

In [ ]:
agg_df.sort_values(by='xcor_mse_0.2_FTT_samplewise')

In [ ]:
peak_mag = agg_df['xcor_peak_magnitude_delta_0.2_FTT_samplewise'].sort_values()
peak_mag[peak_mag>0].dropna()

The top two are Everything_TS=4_prior=0.05 and Everything_TS=16-E=0.25

In [ ]:
agg_df.sort_values(by='xcor_mse_1_FTT_MAP')

Everything_TS=8_no_PDE has the best mse_log_energy_spectrum but it has quite bad xcor...

In [ ]:
agg_df.sort_values(by='mse_log_energy_spectrum_0.2_FTT_samplewise')

# Finding Best Metrics:

In [ ]:
agg_df_MLE.sort_values(by='loss_long')

## Case Study: non-dynamic modes failure
Which metrics are best predictive of the failure? 
* Short term > long term!!
* Also sMAPE super metric? Predictive of xcor & failure case study

In [ ]:
case_study_df = agg_df.drop(columns=['ECE', 'LPPC_val', 'LPPC_val_long', 'Bayesian']).apply(np_normalize).filter(like='MLE+LR_TS=4_K=20', axis=0)
display(case_study_df)

In [ ]:
# bad-to-good (relative) delta: negative is good (assuming minimizer metrics)
((case_study_df.loc['MLE+LR_TS=4_K=20-dyn'] - case_study_df.loc['MLE+LR_TS=4_K=20'])/case_study_df.loc['MLE+LR_TS=4_K=20-dyn'].abs()).sort_values()

In [ ]:
normalized_loss_df.dropna().corr()[['abs_xcor_peak_magnitude_delta_1_FTT_samplewise','xcor_mse_1_FTT_samplewise']].sort_values(by='abs_xcor_peak_magnitude_delta_1_FTT_samplewise', ascending=False)

## sMAPE is most predictive of xcor_mse!

In [ ]:
normalized_loss_df.corr().loc['xcor_mse_1_FTT_samplewise'].sort_values(ascending=False)

In [ ]:
normalized_loss_df.corr().filter(like='sMAPE')

In [ ]:
agg_df.sort_values(by='sMAPE_long')